# 경사 하강법 (Gradient Descent Method)

이 노트북에서는 **경사 하강법(Gradient Descent)**을 구현하고 시각화합니다.
함수 $f(x_0, x_1) = x_0^2 + x_1^2$의 최소점을 찾는 과정을 2차원 평면에 표현합니다.

## 학습 목표

1. **수치 미분(Numerical Differentiation)**을 통한 그래디언트 계산
2. **경사 하강법(Gradient Descent)** 구현
3. **2차원 평면**에서의 최적화 과정 시각화
4. **신경망에서의 기울기** 이해와 구체적인 예시

## 1. 설정 및 준비

필수 라이브러리를 import하고 시각화 설정을 합니다.

In [ ]:
# coding: utf-8
import os, sys
print("현재 작업 디렉토리:", os.getcwd())

import numpy as np
import matplotlib.pyplot as plt

# matplotlib 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

## 2. 수치 미분 (Numerical Differentiation)

### 2.1 개념

**수치 미분**은 매우 작은 값 $h$를 사용하여 미분계수를 근사하는 방법입니다.

**중심 차분법(Central Difference)**:
$$\frac{\partial f}{\partial x} \approx \frac{f(x+h) - f(x-h)}{2h}$$

여기서 $h$는 일반적으로 $10^{-4}$ 정도의 작은 값입니다.

### 2.2 구현

In [ ]:
# 수치미분을 계산하는 원소 함수 (내부 사용)
# 변수가 1개일 때의 수치미분 수행

def _numerical_gradient_no_batch(f, x):
    """
    배치(batch)가 없는 배열에 대한 수치 미분 계산 (내부 사용 함수)
    
    중심 차분(Central Difference) 방법을 사용하여
    각 요소에서의 편미분 값을 계산합니다.
    
    수식: df/dx ≈ (f(x+h) - f(x-h)) / (2h)
    
    Parameters
    ----------
    f : 미분할 함수 (numpy array를 입력받아 스칼라를 반환)
    x : 입력 배열 (numpy array)
    
    Returns
    -------
    numpy array : 각 요소에서의 편미분 값 (기울기)
    """
    h = 1e-4  # 미분 간격 (0.0001) - 충분히 작지만 0은 아님
    grad = np.zeros_like(x)  # x와 동일한 shape의 0 배열 생성 (기울기 저장용)
    
    # 배열의 모든 요소(idx)에 대해 반복
    for idx in range(x.size):
        tmp_val = x[idx]  # 현재 값 백업 (나중에 복원하기 위함)
        
        # f(x+h) 계산: x[idx]에 h를 더한 값으로 함수 계산
        x[idx] = float(tmp_val) + h
        fxh1 = f(x)
        
        # f(x-h) 계산: x[idx]에서 h를 뺀 값으로 함수 계산
        x[idx] = tmp_val - h
        fxh2 = f(x)
        
        # 중심 차분으로 기울기 계산: (f(x+h) - f(x-h)) / (2h)
        grad[idx] = (fxh1 - fxh2) / (2*h)
        
        x[idx] = tmp_val  # 원래 값 복원 (원본 배열 보존)
    
    return grad


# 테스트: 2차원 함수 f(x0, x1) = x0² + x1²
# 해석적 그래디언트: [2*x0, 2*x1]
x_test = np.array([3.0, 4.0])
numerical_grad = _numerical_gradient_no_batch(lambda x: np.sum(x**2), x_test)
analytical_grad = np.array([2 * x_test[0], 2 * x_test[1]])

print(f"=== 다변수 수치 미분 테스트 ===")
print(f"f(x0, x1) = x0² + x1²")
print(f"x = {x_test}")
print(f"수치 미분 그래디언트: {numerical_grad}")
print(f"해석적 그래디언트:   {analytical_grad}")
print(f"오차: {np.abs(numerical_grad - analytical_grad).max():.6f}")

### 2.3 일반적인 수치 미분 함수

변수가 1개일 때와 2개 이상일 때를 모두 처리할 수 있는 일반적인 함수를 구현합니다.

In [ ]:
# 변수가 1개일 때와 2개 이상일 때의 수치미분 수행

def numerical_gradient(f, X):
    """
    수치 미분을 사용하여 함수 f의 X에서의 기울기(그래디언트)를 계산
    
    1차원 배열과 2차원 배열 모두 처리:
    - 1차원: 단일 샘플에 대한 그래디언트 계산
    - 2차원: 여러 샘플에 대한 그래디언트 계산 (각 행별로 계산)
    
    수식: df/dx_i ≈ (f(x_i + h) - f(x_i - h)) / (2h)
    
    Parameters
    ----------
    f : 미분할 함수 (numpy array를 입력받아 스칼라 또는 배열을 반환)
    X : 입력 데이터 (1차원 또는 2차원 numpy array)
    
    Returns
    -------
    numpy array : 기울기 (X와 동일한 shape)
    """
    if X.ndim == 1:
        # 변수가 1개일 때 (1차원 배열): 단일 샘플 처리
        return _numerical_gradient_no_batch(f, X)
    else:
        # 변수가 2개 이상일 때 (2차원 배열): 배치 처리
        # X.shape = (n_samples, n_features) 형태에서
        # 각 샘플(row)별로 그래디언트를 계산
        grad = np.zeros_like(X)  # X와 동일한 shape의 0 배열 생성
        
        # 각 샘플(row)별로 그래디언트 계산
        for idx, x in enumerate(X):
            grad[idx] = _numerical_gradient_no_batch(f, x)
        
        return grad


# 2차원 함수 정의: f(x0, x1) = x0² + x1²
def function_2(x):
    """
    2차원 함수 f(x0, x1) = x0² + x1²
    
    numpy 배열 처리를 위해:
    - 1차원 입력: np.sum(x**2) - 스칼라 반환
    - 2차원 입력: np.sum(x**2, axis=1) - 각 행별 합 반환
    
    해석적 그래디언트: [2*x0, 2*x1]
    """
    if x.ndim == 1:
        # 변수가 하나일 때 (단일 샘플)
        # x = [x0, x1] → x0² + x1² (스칼라 값 반환)
        return np.sum(x**2)
    else:
        # 변수가 두개일 때 (배치 샘플)
        # X.shape = (n, 2) → 각 행별 x0² + x1² 계산
        return np.sum(x**2, axis=1)


# 테스트
x_single = np.array([3.0, 4.0])
grad_single = numerical_gradient(function_2, x_single)
print(f"단일 샘플 테스트:")
print(f"  x = {x_single}")
print(f"  수치 그래디언트: {grad_single}")
print(f"  해석 그래디언트: [2*3.0, 2*4.0] = [6.0, 8.0]")

## 3. 경사 하강법 (Gradient Descent)

### 3.1 개념

**경사 하강법**은 그래디언트(기울기)의 반대 방향으로 단계별로 이동하여 함수의 최소점을 찾는 알고리즘입니다.

**업데이트 공식**:
$$x_{new} = x_{old} - \eta \cdot \nabla f(x)$$

여기서 $\eta$는 **학습률(Learning Rate)**로, 한 단계당 이동하는 크기를 결정합니다.

- 학습률이 너무 크면: 최소점을 지나쳐 발산할 수 있음
- 학습률이 너무 작으면: 수렴하는 데 너무 오래 걸림

### 3.2 구현

In [ ]:
# 수치미분으로 구한 기울기를 이용해 중심으로 이동

def gradient_descent(f, init_x, lr=0.01, step_num=100):
    """
    경사 하강법(Gradient Descent) 구현
    
    그래디언트의 반대 방향으로 단계별로 이동하여 최소점을 찾습니다.
    
    업데이트 공식: x = x - lr * grad
    
    Parameters
    ----------
    f : 미분할 함수 (기울기를 찾을 함수)
    init_x : 초기점 (numpy array)
    lr : 학습률 (learning rate), 기본값 0.01
    step_num : 반복 횟수, 기본값 100
    
    Returns
    -------
    numpy array : 최종 점 (최소점 근사)
    list : 경로 기록 (각 단계의 위치)
    """
    x = init_x.copy()  # 초기점 복사 (원본 보존)
    x_history = [x.copy()]  # 경로 기록 (시각화용)
    
    for i in range(step_num):
        # 현재 점에서의 그래디언트 계산
        grad = numerical_gradient(f, x)
        
        # 그래디언트 반대 방향으로 업데이트
        # x_new = x_old - learning_rate * gradient
        x = x - lr * grad
        
        # 경로 기록
        x_history.append(x.copy())
    
    return np.array(x_history), x_history


# 테스트: f(x0, x1) = x0² + x1²
init_x = np.array([-3.0, 4.0])
lr = 0.1
step_num = 20

x_history, _ = gradient_descent(function_2, init_x, lr=lr, step_num=step_num)

print(f"=== 경사 하강법 시뮬레이션 ===")
print(f"초기점: {init_x}")
print(f"학습률: {lr}")
print(f"반복횟수: {step_num}")
print(f"최종점: {x_history[-1]}")

### 3.3 2차원 평면에서의 시각화

기울기값을 이용해 중심으로 이동시키는 과정을 시각화합니다.

In [ ]:
# 경사 하강법 경로 시각화

# 초기 설정
init_x = np.array([-3.0, 4.0])
lr = 0.1
step_num = 20

# 경사 하강법 실행
x, x_history = gradient_descent(function_2, init_x, lr=lr, step_num=step_num)

# 2차원 평면에 경로 그리기
plt.figure(figsize=(8, 8))

# 원점과 축 표시
plt.plot([-5, 5], [0, 0], '--b', linewidth=1)  # x0 축
plt.plot([0, 0], [-5, 5], '--b', linewidth=1)  # x1 축

# 경사 하강법 경로 (모든 단계)
plt.plot(x[:, 0], x[:, 1], 'o-', linewidth=2, markersize=6, label='Path')

# 시작점과 끝점
plt.scatter([init_x[0]], [init_x[1]], c='green', s=150, marker='*', label='Start', zorder=5)
plt.scatter([x[-1, 0]], [x[-1, 1]], c='red', s=150, marker='X', label='End', zorder=5)

# 축 범위와 레이블
plt.xlim(-3.5, 3.5)
plt.ylim(-4.5, 4.5)
plt.xlabel("x0")
plt.ylabel("x1")
plt.title('Gradient Descent: f(x0, x1) = x0² + x1²')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.show()

print(f"시작점: {init_x}")
print(f"최종점: {x[-1]}")
print(f"함수값: {function_2(x[-1]):.6f}")

## 4. 등고선(Contour)과 함께 시각화

함수의 지형(등고선)과 그래디언트 필드, 최적화 경로를 함께 표시합니다.

In [ ]:
# 등고선(Contour)과 그래디언트 필드, 최적화 경로 함께 표시

# 2차원 손실 지형(콘투어) 생성
x0_range = np.linspace(-3, 3, 200)
x1_range = np.linspace(-3, 3, 200)
X0, X1 = np.meshgrid(x0_range, x1_range)
Z = X0**2 + X1**2  # f(x0, x1) = x0² + x1²

fig, ax = plt.subplots(figsize=(10, 8))

# 등고선(콘투어) 플롯: 함수의 지형 표시
contour = ax.contour(X0, X1, Z, levels=20, cmap='hot_r')
ax.clabel(contour, inline=True, fontsize=6, fmt='%.1f')

# 그래디언트 필드 (약 1/4 간격으로 표시)
skip = 4
X0_sub = X0[::skip, ::skip]
X1_sub = X1[::skip, ::skip]
grad0_sub = 2 * X0_sub  # ∂f/∂x0 = 2*x0
grad1_sub = 2 * X1_sub  # ∂f/∂x1 = 2*x1
ax.quiver(X0_sub, X1_sub, -grad0_sub, -grad1_sub, angles="xy", color='blue', 
          alpha=0.4, scale=50, width=0.002)

# 경사 하강법 경로
init_x = np.array([-3.0, 4.0])
x, x_history = gradient_descent(function_2, init_x, lr=0.1, step_num=20)
ax.plot(x[:, 0], x[:, 1], 'r-', linewidth=2, label='Gradient Descent Path')
ax.plot(x[0, 0], x[0, 1], 'go', markersize=12, label='Start Point')
ax.plot(x[-1, 0], x[-1, 1], 'rx', markersize=15, markeredgewidth=3, label='Converged Point')

# 이론적 최소점
ax.plot(0, 0, 'b*', markersize=20, label='Global Minimum (0, 0)')

ax.set_xlabel('x0', fontsize=12)
ax.set_ylabel('x1', fontsize=12)
ax.set_title('Gradient Descent on f(x0, x1) = x0² + x1²', fontsize=13)
ax.legend(loc='upper right', fontsize=10)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# 신경망에서 기울기 (Gradient in Neural Networks)

## 5. 신경망의 학습 과정

**신경망에서 기울기**는 손실함수(Loss Function)의 값을 줄이는 방향으로 가중치(Weight)를 조정하는 과정입니다.

**가중치 업데이트 공식**:
$$w^+ = w - \eta \cdot \frac{\partial E}{\partial w}$$

| 기호 | 의미 |
|------|------|
| $w$ | 기존 가중치 (Old Weight) |
| $w^+$ | 새롭게 조정된 가중치 (New Weight) |
| $\eta$ | 학습률 (Learning Rate) |
| $\frac{\partial E}{\partial w}$ | 손실함수 기울기값 (Gradient of Loss) |

신경망의 기존 가중치에 기울기를 반영하여 새롭게 조정된 가중치를 구하는 과정이 **학습 과정**입니다.

## 6. 구체적인 예시: 2층 신경망 구현

In [ ]:
# 간단한 2층 신경망 구현
# 입력층 (2개 뉴런) -> 은닉층 (3개 뉴런) -> 출력층 (2개 뉴런)

class SimpleNet:
    """
    간단한 2층 신경망 (Perceptron with 2 hidden neurons)
    
    구조:
    입력층 (2개) -> 은닉층 (3개, ReLU 활성화) -> 출력층 (2개, Softmax)
    
    가중치:
    W1: 입력 -> 은닉 (2x3 행렬)
    b1: 은닉층 바이어스 (3개)
    W2: 은닉 -> 출력 (3x2 행렬)
    b2: 출력층 바이어스 (2개)
    """
    
    def __init__(self):
        # 초기 가중치 설정 (임의의 값)
        self.W1 = np.array([[0.5, -0.3, 0.2],
                           [0.1, 0.4, -0.2]])
        self.b1 = np.array([0.1, -0.1, 0.0])
        self.W2 = np.array([[0.3, -0.1],
                           [0.2, 0.0],
                           [-0.1, 0.3]])
        self.b2 = np.array([0.0, 0.1])
    
    def forward(self, x):
        """
        순전파 (Forward Propagation)
        
        입력 x로부터 출력 계산
        """
        # 은닉층 계산: z1 = x @ W1 + b1
        z1 = np.dot(x, self.W1) + self.b1
        # ReLU 활성화: a1 = ReLU(z1)
        a1 = np.maximum(0, z1)
        
        # 출력층 계산: z2 = a1 @ W2 + b2
        z2 = np.dot(a1, self.W2) + self.b2
        # Softmax 활성화: y = softmax(z2)
        exp_z2 = np.exp(z2 - np.max(z2))
        y = exp_z2 / np.sum(exp_z2)
        
        return y
    
    def loss(self, x, t):
        """
        손실 계산 (Cross-Entropy Error)
        """
        y = self.forward(x)
        # 교차 엔트로피 오차: L = -sum(t * log(y))
        return -np.sum(t * np.log(y + 1e-10))
    
    def accuracy(self, x, t):
        """
        정확도 계산
        """
        y = self.forward(x)
        pred = np.argmax(y)
        true = np.argmax(t)
        return 1.0 if pred == true else 0.0


# 네트워크 인스턴스 생성
net = SimpleNet()

# 입력 데이터와 정답 레이블
x = np.array([0.5, 0.1])
t = np.array([1.0, 0.0])

# 순전파 및 손실 계산
output = net.forward(x)
loss_val = net.loss(x, t)
acc = net.accuracy(x, t)

print(f"=== 간단한 신경망 ===")
print(f"입력: {x}")
print(f"출력 (Softmax 확률): {output}")
print(f"손실 (Loss): {loss_val:.4f}")
print(f"정답 클래스: 0, 예측 클래스: {np.argmax(output)}")
print(f"정확도: {acc:.0%}")

print(f"\n=== 가중치 초기값 ===")
print(f"W1 shape: {net.W1.shape}")
print(f"W1:\n{net.W1}")
print(f"b1: {net.b1}")
print(f"W2 shape: {net.W2.shape}")
print(f"W2:\n{net.W2}")
print(f"b2: {net.b2}")

## 7. 가중치 기울기 계산

각 가중치에 대한 손실함수의 기울기를 수치 미분으로 계산합니다.

In [ ]:
# 가중치 기울기 계산
# 각 가중치에 대한 손실함수 기울기를 수치 미분으로 계산

# 수치 미분을 위한 작은 값 h
h = 1e-4

# W1에 대한 기울기 계산
# 각 요소 W1[i,j]별로 수치 미분 수행
grad_W1 = np.zeros_like(net.W1)
for i in range(net.W1.shape[0]):
    for j in range(net.W1.shape[1]):
        # 현재 값 백업
        original = net.W1[i, j]
        
        # f(x+h) 계산: W1[i,j]에 h를 더했을 때 손실
        net.W1[i, j] = original + h
        loss_plus = net.loss(x, t)
        
        # f(x-h) 계산: W1[i,j]에서 h를 뺐을 때 손실
        net.W1[i, j] = original - h
        loss_minus = net.loss(x, t)
        
        # 중심 차분으로 기울기 계산: (f(x+h) - f(x-h)) / (2h)
        grad_W1[i, j] = (loss_plus - loss_minus) / (2 * h)
        
        # 원래 값 복원
        net.W1[i, j] = original


# W2에 대한 기울기 계산
grad_W2 = np.zeros_like(net.W2)
for i in range(net.W2.shape[0]):
    for j in range(net.W2.shape[1]):
        # 현재 값 백업
        original = net.W2[i, j]
        
        # f(x+h) 계산: W2[i,j]에 h를 더했을 때 손실
        net.W2[i, j] = original + h
        loss_plus = net.loss(x, t)
        
        # f(x-h) 계산: W2[i,j]에서 h를 뺐을 때 손실
        net.W2[i, j] = original - h
        loss_minus = net.loss(x, t)
        
        # 중심 차분으로 기울기 계산: (f(x+h) - f(x-h)) / (2h)
        grad_W2[i, j] = (loss_plus - loss_minus) / (2 * h)
        
        # 원래 값 복원
        net.W2[i, j] = original


print(f"=== 가중치 기울기 계산 ===")
print(f"입력: {x}, 정답: {t}")
print(f"\nW1 = ")
print(net.W1)
print(f"\ndL/dW1 (W1의 각 요소에 대한 기울기) = ")
print(grad_W1)
print(f"\nW2 = ")
print(net.W2)
print(f"\ndL/dW2 (W2의 각 요소에 대한 기울기) = ")
print(grad_W2)

# 단일 요소 예시 확인
print(f"\n=== 단일 요소 확인 ===")
print(f"W1[0, 0] = {net.W1[0, 0]}")
print(f"dL/dW1[0,0] = {grad_W1[0, 0]:.4f}")
print(f"\n기울기 의미: W1[0,0]을 h만큼 증가시키면 손실이 {grad_W1[0, 0] * h:.6f} 증가함")
print(f"따라서 가중치 업데이트: W1[0,0] -= lr * grad_W1[0,0]로 감소시킴")

## 8. 가중치 업데이트 시각화

기울기를 사용하여 가중치를 업데이트하는 과정을 시각화합니다.

In [ ]:
# 가중치 업데이트 과정 시각화
# 기울기를 반영하여 가중치를 조정하는 과정

# 네트워크 재초기화 (안정적인 학습을 위해)
net_train = SimpleNet()

# 학습 설정
lr = 0.1
step_num = 100

# 손실 기록
loss_history = []
loss_history.append(net_train.loss(x, t))

print(f"=== 가중치 업데이트 과정 ===")
print(f"학습률: {lr}")
print(f"반복횟수: {step_num}")
print(f"초기 손실: {loss_history[0]:.4f}")

for i in range(step_num):
    # W1 업데이트
    grad_W1 = np.zeros_like(net_train.W1)
    for ii in range(net_train.W1.shape[0]):
        for jj in range(net_train.W1.shape[1]):
            h = 1e-4
            original = net_train.W1[ii, jj]
            
            net_train.W1[ii, jj] = original + h
            loss_plus = net_train.loss(x, t)
            
            net_train.W1[ii, jj] = original - h
            loss_minus = net_train.loss(x, t)
            
            grad_W1[ii, jj] = (loss_plus - loss_minus) / (2 * h)
            net_train.W1[ii, jj] = original
    
    net_train.W1 = net_train.W1 - lr * grad_W1
    
    # W2 업데이트
    grad_W2 = np.zeros_like(net_train.W2)
    for ii in range(net_train.W2.shape[0]):
        for jj in range(net_train.W2.shape[1]):
            h = 1e-4
            original = net_train.W2[ii, jj]
            
            net_train.W2[ii, jj] = original + h
            loss_plus = net_train.loss(x, t)
            
            net_train.W2[ii, jj] = original - h
            loss_minus = net_train.loss(x, t)
            
            grad_W2[ii, jj] = (loss_plus - loss_minus) / (2 * h)
            net_train.W2[ii, jj] = original
    
    net_train.W2 = net_train.W2 - lr * grad_W2
    
    # 손실 기록
    loss_history.append(net_train.loss(x, t))
    
    if (i + 1) % 20 == 0:
        print(f"  Step {i+1}: Loss = {loss_history[-1]:.4f}")

print(f"최종 손실: {loss_history[-1]:.4f}")

# 손실 감소 시각화
plt.figure(figsize=(10, 6))
plt.plot(range(len(loss_history)), loss_history, 'o-', linewidth=2, markersize=4)
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Loss (Cross-Entropy Error)', fontsize=12)
plt.title('Loss Reduction During Weight Update (Gradient Descent)', fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. 가중치 기울기 분포 시각화

각 가중치의 기울기 값 분포를 히스토그램으로 확인합니다.

In [ ]:
# 가중치와 기울기 분포 시각화

# 현재 가중치 기준 기울기 계산
net_vis = SimpleNet()

grad_W1 = np.zeros_like(net_vis.W1)
for i in range(net_vis.W1.shape[0]):
    for j in range(net_vis.W1.shape[1]):
        h = 1e-4
        original = net_vis.W1[i, j]
        net_vis.W1[i, j] = original + h
        loss_plus = net_vis.loss(x, t)
        net_vis.W1[i, j] = original - h
        loss_minus = net_vis.loss(x, t)
        grad_W1[i, j] = (loss_plus - loss_minus) / (2 * h)
        net_vis.W1[i, j] = original

grad_W2 = np.zeros_like(net_vis.W2)
for i in range(net_vis.W2.shape[0]):
    for j in range(net_vis.W2.shape[1]):
        h = 1e-4
        original = net_vis.W2[i, j]
        net_vis.W2[i, j] = original + h
        loss_plus = net_vis.loss(x, t)
        net_vis.W2[i, j] = original - h
        loss_minus = net_vis.loss(x, t)
        grad_W2[i, j] = (loss_plus - loss_minus) / (2 * h)
        net_vis.W2[i, j] = original

# 시각화
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# W1 분포
axes[0, 0].hist(net_vis.W1.flatten(), bins=10, color='steelblue', edgecolor='black')
axes[0, 0].set_title('W1 Weight Distribution', fontsize=12)
axes[0, 0].set_xlabel('Weight Value')
axes[0, 0].set_ylabel('Count')

# W1 기울기 분포
axes[0, 1].hist(grad_W1.flatten(), bins=10, color='coral', edgecolor='black')
axes[0, 1].set_title('W1 Gradient Distribution (dL/dW1)', fontsize=12)
axes[0, 1].set_xlabel('Gradient Value')
axes[0, 1].set_ylabel('Count')

# W2 분포
axes[1, 0].hist(net_vis.W2.flatten(), bins=10, color='steelblue', edgecolor='black')
axes[1, 0].set_title('W2 Weight Distribution', fontsize=12)
axes[1, 0].set_xlabel('Weight Value')
axes[1, 0].set_ylabel('Count')

# W2 기울기 분포
axes[1, 1].hist(grad_W2.flatten(), bins=10, color='coral', edgecolor='black')
axes[1, 1].set_title('W2 Gradient Distribution (dL/dW2)', fontsize=12)
axes[1, 1].set_xlabel('Gradient Value')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"=== 가중치 및 기울기 요약 ===")
print(f"W1: mean={np.mean(net_vis.W1):.4f}, std={np.std(net_vis.W1):.4f}")
print(f"dL/dW1: mean={np.mean(grad_W1):.4f}, std={np.std(grad_W1):.4f}")
print(f"W2: mean={np.mean(net_vis.W2):.4f}, std={np.std(net_vis.W2):.4f}")
print(f"dL/dW2: mean={np.mean(grad_W2):.4f}, std={np.std(grad_W2):.4f}")

## 10. 여러 입력에 대한 학습 과정 시각화

여러 학습 데이터를 사용하여 신경망을 학습시키는 과정을 시각화합니다.

In [ ]:
# 여러 입력에 대한 학습 과정 시각화

# 학습 데이터 (4개 샘플)
train_data = [
    (np.array([0.5, 0.1]), np.array([1.0, 0.0])),
    (np.array([0.1, 0.5]), np.array([0.0, 1.0])),
    (np.array([0.8, 0.2]), np.array([1.0, 0.0])),
    (np.array([0.2, 0.8]), np.array([0.0, 1.0])),
]

# 네트워크 재초기화
net2 = SimpleNet()

# 학습 설정
lr = 0.5
step_num = 100
loss_history = []
acc_history = []

# 초기 손실 및 정확도
total_loss = sum(net2.loss(xd, td) for xd, td in train_data)
total_acc = sum(net2.accuracy(xd, td) for xd, td in train_data)
loss_history.append(total_loss / len(train_data))
acc_history.append(total_acc / len(train_data))

print(f"=== 신경망 학습 과정 ===")
print(f"학습 데이터: {len(train_data)} 샘플")
print(f"학습률: {lr}")
print(f"반복횟수: {step_num}")
print(f"초기 손실: {loss_history[0]:.4f}, 초기 정확도: {acc_history[0]:.0%}")

for i in range(step_num):
    # 각 가중치 기울기 초기화
    grad_W1 = np.zeros_like(net2.W1)
    grad_W2 = np.zeros_like(net2.W2)
    
    # 전체 학습 데이터에 대한 평균 기울기
    for x_data, t_data in train_data:
        # W1 기울기
        for ii in range(net2.W1.shape[0]):
            for jj in range(net2.W1.shape[1]):
                h = 1e-4
                original = net2.W1[ii, jj]
                net2.W1[ii, jj] = original + h
                loss_plus = net2.loss(x_data, t_data)
                net2.W1[ii, jj] = original - h
                loss_minus = net2.loss(x_data, t_data)
                grad_W1[ii, jj] += (loss_plus - loss_minus) / (2 * h)
                net2.W1[ii, jj] = original
        
        # W2 기울기
        for ii in range(net2.W2.shape[0]):
            for jj in range(net2.W2.shape[1]):
                h = 1e-4
                original = net2.W2[ii, jj]
                net2.W2[ii, jj] = original + h
                loss_plus = net2.loss(x_data, t_data)
                net2.W2[ii, jj] = original - h
                loss_minus = net2.loss(x_data, t_data)
                grad_W2[ii, jj] += (loss_plus - loss_minus) / (2 * h)
                net2.W2[ii, jj] = original
    
    # 평균 기울기
    n = len(train_data)
    grad_W1 /= n
    grad_W2 /= n
    
    # 가중치 업데이트
    net2.W1 -= lr * grad_W1
    net2.W2 -= lr * grad_W2
    
    # 손실 및 정확도 기록
    total_loss = sum(net2.loss(xd, td) for xd, td in train_data)
    total_acc = sum(net2.accuracy(xd, td) for xd, td in train_data)
    loss_history.append(total_loss / n)
    acc_history.append(total_acc / n)
    
    if (i + 1) % 20 == 0:
        print(f"  Step {i+1}: Loss = {loss_history[-1]:.4f}, Accuracy = {acc_history[-1]:.0%}")

# 학습 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 손실 곡선
axes[0].plot(range(len(loss_history)), loss_history, 'o-', linewidth=2, markersize=4)
axes[0].set_xlabel('Training Step', fontsize=12)
axes[0].set_ylabel('Average Loss', fontsize=12)
axes[0].set_title('Training Loss Over Time', fontsize=13)
axes[0].grid(True, alpha=0.3)

# 정확도 곡선
axes[1].plot(range(len(acc_history)), acc_history, 'o-', linewidth=2, markersize=4, color='green')
axes[1].set_xlabel('Training Step', fontsize=12)
axes[1].set_ylabel('Average Accuracy', fontsize=12)
axes[1].set_title('Training Accuracy Over Time', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. 요약 및 정리

### 경사 하강법 핵심 개념

| 개념 | 설명 |
|------|------|
| **기울기 (Gradient)** | 함수가 가장 빠르게 증가하는 방향 |
| **경사 하강법** | 기울기의 반대 방향으로 이동하여 최소점 찾기 |
| **학습률 (Learning Rate)** | 한 단계당 이동 크기 ($\eta$) |
| **수치 미분** | $\frac{\partial f}{\partial x} \approx \frac{f(x+h) - f(x-h)}{2h}$ |

### 신경망 학습 과정

1. **순전파 (Forward)**: 입력으로부터 출력과 손실 계산
2. **기울기 계산**: 손실함수의 가중치에 대한 기울기 계산
3. **가중치 업데이트**: $w^+ = w - \eta \cdot \frac{\partial L}{\partial w}$
4. **반복**: 손실이 충분히 작아질 때까지 1-3 단계 반복

### 학습률의 영향

| 학습률 | 결과 |
|--------|------|
| 너무 큼 | 발산하거나 진동 |
| 적절함 | 안정적으로 수렴 |
| 너무 작음 | 수렴에 너무 오래 걸림 |